# MiMo-V2.5 and MiMo-V2.5-Pro Cookbook

This notebook is a single cookbook for both Xiaomi MiMo-V2.5 and MiMo-V2.5-Pro using the OpenAI-compatible API.

What is covered:
- setup and auth
- basic chat completions for both models
- streaming with reasoning output
- framework integration (LangChain)
- tool-enabled agent loop with reasoning carryover
- long-context processing pattern
- structured JSON response pattern
- multimodal input pattern (MiMo-V2.5)
- token usage tracking

Sources:
- https://mimo.xiaomi.com/mimo-v2-5
- https://mimo.xiaomi.com/mimo-v2-5-pro
- https://platform.xiaomimimo.com/docs/quick-start/first-api-call
- https://platform.xiaomimimo.com/docs/api/chat/openai-api
- https://platform.xiaomimimo.com/docs/pricing

## 0) Setup and Installation

In [ ]:
# Run this once in a fresh environment.
# %pip install -U openai langchain langchain-openai

In [ ]:
import os
from openai import OpenAI

MODEL_STANDARD = "mimo-v2.5"
MODEL_PRO = "mimo-v2.5-pro"
MODEL = os.getenv("MIMO_MODEL", MODEL_STANDARD)
BASE_URL = os.getenv("MIMO_BASE_URL", "https://api.xiaomimimo.com/v1")
API_KEY = os.getenv("MIMO_API_KEY")

if MODEL not in {MODEL_STANDARD, MODEL_PRO}:
    raise ValueError("MIMO_MODEL must be either 'mimo-v2.5' or 'mimo-v2.5-pro'.")

if not API_KEY:
    raise ValueError("Set MIMO_API_KEY before running this notebook.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print(f"Using model: {MODEL}")
print(f"Base URL: {BASE_URL}")

In [ ]:
def pick_model(use_pro: bool = False) -> str:
    return MODEL_PRO if use_pro else MODEL_STANDARD

print("Standard:", pick_model(False))
print("Pro:", pick_model(True))

## 1) Basic Usage with Provider SDK

In [ ]:
def run_once(model_name: str, prompt: str):
    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a concise coding assistant."},
            {"role": "user", "content": prompt},
        ],
        thinking={"type": "enabled"},
        max_completion_tokens=1024,
    )
    print(f"\n=== {model_name} ===")
    print(resp.choices[0].message.content)

prompt = "List 5 release checks before enabling autonomous code actions in production."
run_once(MODEL_STANDARD, prompt)
run_once(MODEL_PRO, prompt)

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain canary deploy rollback in one short example."}],
    thinking={"type": "enabled"},
    stream=True,
)

seen_answer = False
print("\n" + "=" * 20 + " Reasoning " + "=" * 20)
for chunk in stream:
    delta = chunk.choices[0].delta
    reasoning = getattr(delta, "reasoning_content", None)
    content = getattr(delta, "content", None)

    if reasoning and not seen_answer:
        print(reasoning, end="", flush=True)

    if content:
        if not seen_answer:
            seen_answer = True
            print("\n" + "=" * 20 + " Answer " + "=" * 20)
        print(content, end="", flush=True)
print()

### Preserve reasoning content across turns

MiMo docs recommend keeping prior `reasoning_content` messages during multi-turn tool workflows for best performance.

In [ ]:
messages = [
    {"role": "system", "content": "You are a senior backend engineer."},
    {"role": "user", "content": "Draft a webhook retry strategy for payment events."},
]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    thinking={"type": "enabled"},
)

first_message = first.choices[0].message
messages.append({
    "role": "assistant",
    "content": first_message.content or "",
    "reasoning_content": getattr(first_message, "reasoning_content", None),
})
messages.append({"role": "user", "content": "Now adapt it for strict latency SLOs."})

second = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    thinking={"type": "enabled"},
)
print(second.choices[0].message.content)

## 2) Framework Integration

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.8,
)

msg = llm.invoke("Generate a rollout checklist for adding a new model into a CI-based eval gate.")
print(msg.content)

## 3) Building Agents

In [ ]:
import json

def check_service_dependency(name: str):
    status = {"postgres": "ok", "redis": "ok", "legacy-cache": "deprecated"}
    return {"dependency": name, "status": status.get(name, "unknown")}

tools = [
    {
        "type": "function",
        "function": {
            "name": "check_service_dependency",
            "description": "Return health status for a dependency.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string"}},
                "required": ["name"],
            },
        },
    }
]

In [ ]:
messages = [
    {"role": "system", "content": "You are a release assistant. Use tools before deciding."},
    {"role": "user", "content": "Can we deploy if the service depends on legacy-cache?"},
]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    thinking={"type": "enabled"},
)

assistant_message = first.choices[0].message
messages.append({
    "role": "assistant",
    "content": assistant_message.content or "",
    "tool_calls": assistant_message.tool_calls,
    "reasoning_content": getattr(assistant_message, "reasoning_content", None),
})

if assistant_message.tool_calls:
    for call in assistant_message.tool_calls:
        if call.function.name == "check_service_dependency":
            args = json.loads(call.function.arguments)
            result = check_service_dependency(args["name"])
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })

final = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    thinking={"type": "enabled"},
)
print(final.choices[0].message.content)

## 4) Advanced Applications

In [ ]:
def chunk_text(text: str, chunk_size: int = 3000):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

# Replace with your real long document.
long_doc = """
MiMo-V2.5 and MiMo-V2.5-Pro support long-horizon agentic workflows.
Use chunking plus selective retrieval for stable long-context QA.
""" * 300

chunks = chunk_text(long_doc)
context = "\n\n".join(chunks[:40])

qa = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer using only the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: What rollout steps should an engineering team prioritize first?"},
    ],
    max_completion_tokens=2048,
)
print(qa.choices[0].message.content)

In [ ]:
json_resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Return valid JSON only."},
        {"role": "user", "content": "Provide a deploy checklist with fields: risk, owner, mitigation. Return JSON array with 3 items."},
    ],
    response_format={"type": "json_object"},
)

payload = json_resp.choices[0].message.content
print(payload)

In [ ]:
# MiMo-V2.5 has native multimodal understanding. Keep this on mimo-v2.5 unless your account docs confirm pro multimodal support.
multimodal_model = MODEL_STANDARD

# Replace IMAGE_URL with your own accessible image URL.
IMAGE_URL = "https://images.unsplash.com/photo-1461749280684-dccba630e2f6"

vision = client.chat.completions.create(
    model=multimodal_model,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Summarize the engineering risks visible in this screenshot."},
                {"type": "image_url", "image_url": {"url": IMAGE_URL}},
            ],
        }
    ],
)
print(vision.choices[0].message.content)

In [ ]:
usage_demo = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Give 3 short notes on production safety for AI agents."}],
    max_completion_tokens=512,
)

usage = usage_demo.usage
print("prompt_tokens:", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("total_tokens:", usage.total_tokens)
if getattr(usage, "completion_tokens_details", None):
    print("reasoning_tokens:", usage.completion_tokens_details.reasoning_tokens)

## Model-Specific Notes

- API model ids for this notebook: `mimo-v2.5` and `mimo-v2.5-pro`.
- OpenAI-compatible base URL: `https://api.xiaomimimo.com/v1`.
- Chat endpoint: `https://api.xiaomimimo.com/v1/chat/completions`.
- Auth headers supported by docs:
  - `api-key: $MIMO_API_KEY`
  - `Authorization: Bearer $MIMO_API_KEY`
- Context window (pricing page):
  - MiMo-V2.5: 1M
  - MiMo-V2.5-Pro: 1M
- Maximum output length (pricing page):
  - MiMo-V2.5: 128K
  - MiMo-V2.5-Pro: 128K
- API defaults for max_completion_tokens (OpenAI compatibility doc):
  - MiMo-V2.5: 32768
  - MiMo-V2.5-Pro: 131072
- Token Plan multiplier from release notes:
  - MiMo-V2.5: 1x
  - MiMo-V2.5-Pro: 2x
- Overseas API pricing snapshots from the pricing page:
  - MiMo-V2.5 (Omni series):
    - 0 to 256K context: Input $0.40/M, cached input $0.08/M, output $2.00/M
    - 256K to 1M context: Input $0.80/M, cached input $0.16/M, output $4.00/M
  - MiMo-V2.5-Pro (Pro series):
    - 0 to 256K context: Input $1.00/M, cached input $0.20/M, output $3.00/M
    - 256K to 1M context: Input $2.00/M, cached input $0.40/M, output $6.00/M
- MiMo-V2.5 release page explicitly highlights native multimodal understanding and 1M context.
- MiMo-V2.5-Pro release page explicitly highlights stronger long-range agentic and coding performance.
- Verify model capability availability by region and account tier before production rollout.

---
Cookbook done. Set `MIMO_API_KEY`, pick `MIMO_MODEL` as `mimo-v2.5` or `mimo-v2.5-pro`, then run each section end-to-end in your target environment.